In [ ]:
!pip -q install transformers datasets torch scikit-learn evaluate accelerate

In [ ]:
from datasets import load_dataset

# Chargement du dataset
dataset = load_dataset("liar")

# Aperçu des données
print(dataset['train'][0])
# Les labels vont de 0 à 5 :
# 0: false, 1: half-true, 2: mostly-true, 3: true, 4: barely-true, 5: pants-fire

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    # On tokenise la déclaration (statement) et on peut ajouter le contexte (subject/speaker) si désiré
    return tokenizer(examples['statement'], truncation=True, padding="max_length", max_length=128)

# Application de la tokenisation à tout le dataset
encoded_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = 6  # LIAR a 6 classes
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=num_labels
)

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="bert-liar-finetuned",
    evaluation_strategy="epoch",     # Evaluer à chaque fin d'époque
    save_strategy="epoch",           # Sauvegarder à chaque fin d'époque
    learning_rate=2e-5,              # Taux d'apprentissage standard pour BERT
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,              # 3 à 5 époques suffisent généralement
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Lancement de l'entraînement
trainer.train()

In [ ]:
text = "The economy is growing faster than ever before."

inputs = tokenizer(text, return_tensors="pt").to("cuda") # ou "cpu"
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=-1)

labels_list = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
print(f"Prédiction : {labels_list[predictions.item()]}")